In [17]:
import pandas as pd
import numpy as np
import optuna
import torch
import contextlib
import os
import time
import pickle
import warnings
import random
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator

from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [18]:
df = pd.read_csv("data/Porto Seguro Safe Driver/train.csv")

print(df.shape)
print(df.head())

(595212, 59)
   id  target  ps_ind_01  ps_ind_02_cat  ps_ind_03  ps_ind_04_cat  \
0   7       0          2              2          5              1   
1   9       0          1              1          7              0   
2  13       0          5              4          9              1   
3  16       0          0              1          2              0   
4  17       0          0              2          0              1   

   ps_ind_05_cat  ps_ind_06_bin  ps_ind_07_bin  ps_ind_08_bin  ...  \
0              0              0              1              0  ...   
1              0              0              0              1  ...   
2              0              0              0              1  ...   
3              0              1              0              0  ...   
4              0              1              0              0  ...   

   ps_calc_11  ps_calc_12  ps_calc_13  ps_calc_14  ps_calc_15_bin  \
0           9           1           5           8               0   
1           3

In [19]:
# Missing values are encoded as -1 in this dataset
print("'-1' counts per column (missing values):")
print((df == -1).sum()[df.columns[(df == -1).sum() > 0]])

print("\nDuplicates:", df.duplicated().sum())

print("\nClass distribution:")
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True).round(3))

#print percentage missing values per column, print only columns with missing values
print("\n'-1' percentage per column:")
print(((df == -1).sum() / len(df) * 100).round(2)[df.columns[((df == -1).sum() / len(df) * 100) > 0]])

'-1' counts per column (missing values):
ps_ind_02_cat       216
ps_ind_04_cat        83
ps_ind_05_cat      5809
ps_reg_03        107772
ps_car_01_cat       107
ps_car_02_cat         5
ps_car_03_cat    411231
ps_car_05_cat    266551
ps_car_07_cat     11489
ps_car_09_cat       569
ps_car_11             5
ps_car_12             1
ps_car_14         42620
dtype: int64

Duplicates: 0

Class distribution:
target
0    573518
1     21694
Name: count, dtype: int64
target
0    0.964
1    0.036
Name: proportion, dtype: float64

'-1' percentage per column:
ps_ind_02_cat     0.04
ps_ind_04_cat     0.01
ps_ind_05_cat     0.98
ps_reg_03        18.11
ps_car_01_cat     0.02
ps_car_02_cat     0.00
ps_car_03_cat    69.09
ps_car_05_cat    44.78
ps_car_07_cat     1.93
ps_car_09_cat     0.10
ps_car_11         0.00
ps_car_12         0.00
ps_car_14         7.16
dtype: float64


In [20]:
# Drop id — not a predictive feature
df = df.drop(columns=["id"])

# Identify column groups by naming convention
CAT_COLS = [c for c in df.columns if c.endswith("_cat")]
BIN_COLS = [c for c in df.columns if c.endswith("_bin")]
NUM_COLS = [c for c in df.columns if c not in CAT_COLS + BIN_COLS + ["target"]]

y = df["target"]
X = df.drop(columns=["target"])

print("Categorical columns:", CAT_COLS)
print("Binary columns:", BIN_COLS)
print("Numeric columns:", NUM_COLS)
print("\nTotal features:", X.shape[1])
print("Target distribution:\n", y.value_counts())

#count number of numeric and categorical features
num_numeric = len(NUM_COLS) + len(BIN_COLS)
num_categorical = len(CAT_COLS)
print(f"Number of numeric features: {num_numeric}")
print(f"Number of categorical features: {num_categorical}")

Categorical columns: ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat', 'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat']
Binary columns: ['ps_ind_06_bin', 'ps_ind_07_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin', 'ps_calc_15_bin', 'ps_calc_16_bin', 'ps_calc_17_bin', 'ps_calc_18_bin', 'ps_calc_19_bin', 'ps_calc_20_bin']
Numeric columns: ['ps_ind_01', 'ps_ind_03', 'ps_ind_14', 'ps_ind_15', 'ps_reg_01', 'ps_reg_02', 'ps_reg_03', 'ps_car_11', 'ps_car_12', 'ps_car_13', 'ps_car_14', 'ps_car_15', 'ps_calc_01', 'ps_calc_02', 'ps_calc_03', 'ps_calc_04', 'ps_calc_05', 'ps_calc_06', 'ps_calc_07', 'ps_calc_08', 'ps_calc_09', 'ps_calc_10', 'ps_calc_11', 'ps_calc_12', 'ps_calc_13', 'ps_calc_14']

Total features: 57
Target distribution:
 target
0    5735

We subsample the dataset due to computational constraints

In [21]:
# Subsample to 33% due to computational constraints
# Stratified to preserve the severe class imbalance (~96%/~4%)
X_sub, _, y_sub, _ = train_test_split(
    X, y,
    train_size=0.33, random_state=0,
    stratify=y
)

X = X_sub
y = y_sub

print(f"Subsampled dataset size: {X.shape[0]} rows")
print("Target distribution after subsample:\n", y.value_counts())
print(y.value_counts(normalize=True).round(3))

Subsampled dataset size: 196419 rows
Target distribution after subsample:
 target
0    189260
1      7159
Name: count, dtype: int64
target
0    0.964
1    0.036
Name: proportion, dtype: float64


Check for cardinality

In [22]:
# Check cardinality of categorical columns
print("Categorical column cardinality:")
for col in CAT_COLS:
    print(f"  {col}: {X[col].nunique()} unique values")

#print average cardinality of categorical features
avg_cardinality = np.mean([X[col].nunique() for col in CAT_COLS])
print(f"Average cardinality of categorical features: {avg_cardinality:.1f}")

Categorical column cardinality:
  ps_ind_02_cat: 5 unique values
  ps_ind_04_cat: 3 unique values
  ps_ind_05_cat: 8 unique values
  ps_car_01_cat: 13 unique values
  ps_car_02_cat: 3 unique values
  ps_car_03_cat: 3 unique values
  ps_car_04_cat: 10 unique values
  ps_car_05_cat: 3 unique values
  ps_car_06_cat: 18 unique values
  ps_car_07_cat: 3 unique values
  ps_car_08_cat: 2 unique values
  ps_car_09_cat: 6 unique values
  ps_car_10_cat: 3 unique values
  ps_car_11_cat: 104 unique values
Average cardinality of categorical features: 13.1


**TabNet**
---
Porto Seguro contains a mix of binary, categorical, and numeric features, with missing values encoded as -1 across several columns. 



For categorical features, TabNet requires integer-encoded inputs. It provides native categorical embedding support via cat_idxs and cat_dims, where each category is mapped to a learned dense vector representation. As a result, label encoding (assigning a unique integer to each category) is sufficient, and techniques such as one-hot encoding or target encoding are not required. The "-1" missing values are treated as their own category during label encoding and require no additional processing.


For numeric and binary features, StandardScaler normalization is applied, with the scaler fit on the training set only and applied to validation and test sets to prevent data leakage. For numeric columns containing -1 as a missing value indicator, these are left as-is before scaling — the scaler will normalize them alongside valid values, and the model learns to handle them.

The data is split into 60/20/20 train/validation/test sets using stratified sampling to preserve the severe class imbalance (~96% no claim, ~4% claim). AUC-ROC is the primary metric here as accuracy alone would be misleading given the imbalance.

In [23]:
# Label-encode categoricals — -1 missing values become their own category naturally
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# 60/20/20 split — stratified to preserve severe class imbalance
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0, stratify=y_temp)

# Scale numeric and binary columns — fit on train only
SCALE_COLS = NUM_COLS + BIN_COLS
scaler = StandardScaler()
X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
X_val[SCALE_COLS]   = scaler.transform(X_val[SCALE_COLS])
X_test[SCALE_COLS]  = scaler.transform(X_test[SCALE_COLS])

# Build cat_idxs and cat_dims for TabNet
cat_idxs = [X_train.columns.get_loc(c) for c in CAT_COLS]
cat_dims  = [X[c].nunique() for c in CAT_COLS]

# Convert to numpy arrays for TabNet
X_train_tab = X_train.values.astype(np.float32)
X_val_tab   = X_val.values.astype(np.float32)
X_test_tab  = X_test.values.astype(np.float32)

y_train_arr = y_train.values
y_val_arr   = y_val.values
y_test_arr  = y_test.values

print("cat_idxs:", cat_idxs)
print("cat_dims:", cat_dims)
print("Train shape:", X_train_tab.shape)



cat_idxs: [1, 3, 4, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
cat_dims: [5, 3, 8, 13, 3, 3, 10, 3, 18, 3, 2, 6, 3, 104]
Train shape: (117851, 57)


**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `n_d` / `n_a` (kept equal): controls the width of the decision step
- `n_steps`: number of sequential attention steps
- `lr`: learning rate

In [24]:
def objective(trial):
    n_da = trial.suggest_int("n_da", 8, 32, step=8)
    params = {
        "n_da": n_da,
        "n_steps": trial.suggest_int("n_steps", 3, 10),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    }

    model = TabNetClassifier(
        n_d=params["n_da"],
        n_a=params["n_da"],
        n_steps=params["n_steps"],
        gamma=1.3,
        lambda_sparse=1e-5,
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        optimizer_params=dict(lr=params["lr"]),
        seed=0,
        verbose=0
    )

    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        model.fit(
            X_train_tab, y_train_arr,
            eval_set=[(X_val_tab, y_val_arr)],
            eval_metric=["auc"],
            max_epochs=30,
            patience=3,
            batch_size=1024,
            virtual_batch_size=128
        )

    preds_proba = model.predict_proba(X_val_tab)[:, 1]
    auc = roc_auc_score(y_val_arr, preds_proba)
    return auc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

Best AUC: 0.5935188457455884
Best params: {'n_da': 8, 'n_steps': 3, 'lr': 0.006358799913027765}


**Final Evaluation**

In [25]:
best_params = study.best_params
seeds = [1, 2, 3]

final_acc, final_auc = [], []

for seed in seeds:
    model = TabNetClassifier(
        n_d=best_params["n_da"],
        n_a=best_params["n_da"],
        n_steps=best_params["n_steps"],
        gamma=1.3,
        lambda_sparse=1e-5,
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        optimizer_params=dict(lr=best_params["lr"]),
        seed=seed,
        verbose=0
    )

    with open(os.devnull, "w") as f, contextlib.redirect_stdout(f):
        model.fit(
            X_train_tab, y_train_arr,
            eval_set=[(X_val_tab, y_val_arr)],
            eval_metric=["auc"],
            max_epochs=30,
            patience=3,
            batch_size=1024,
            virtual_batch_size=128
        )

    preds = model.predict(X_test_tab)
    preds_proba = model.predict_proba(X_test_tab)[:, 1]

    final_acc.append(accuracy_score(y_test_arr, preds))
    final_auc.append(roc_auc_score(y_test_arr, preds_proba))

print(f"TabNet Accuracy: {np.mean(final_acc):.4f} ± {np.std(final_acc):.4f}")
print(f"TabNet AUC-ROC:  {np.mean(final_auc):.4f} ± {np.std(final_auc):.4f}")

TabNet Accuracy: 0.9635 ± 0.0000
TabNet AUC-ROC:  0.5634 ± 0.0101


**Inference Cost**

In [26]:
start = time.time()
_ = model.predict(X_test_tab)
elapsed = time.time() - start

print(f"Inference time per sample: {elapsed/len(X_test_tab)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model)) / 1024:.2f} KB")

Inference time per sample: 0.0133 ms
Model size: 427.97 KB


**Export Results**

In [27]:
results = {
    "Model": ["TabNet"],
    "Dataset": ["Porto Seguro"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc)],
    "Accuracy Std": [np.std(final_acc)],
    "AUC-ROC Mean": [np.mean(final_auc)],
    "AUC-ROC Std": [np.std(final_auc)],
    "Inference Time (ms/sample)": [elapsed / len(X_test_tab) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model)) / 1024],
    "Feature Ranking (most to least important)": [
        str(pd.Series(model.feature_importances_, index=X_train.columns)
            .sort_values(ascending=False)
            .index.tolist())
    ]
}

results_df = pd.DataFrame(results)
results_df.to_csv("results/results_porto_seguro.csv", index=False)
print(results_df)

    Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0  TabNet  Porto Seguro  Classification       0.963548           0.0   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0        0.5634     0.010111                    0.013349       427.969727   

           Feature Ranking (most to least important)  
0  ['ps_ind_06_bin', 'ps_calc_17_bin', 'ps_car_12...  


**FT-Transformer**
---
The same train/validation/test split is used.

FT-Transformer is implemented via pytorch-tabular, which accepts pandas DataFrames directly and handles categorical embeddings internally through its DataConfig — categorical columns are passed as raw strings and pytorch-tabular learns embeddings for them natively. As a result, no label encoding is required for FT-Transformer, unlike TabNet. Missing values encoded as -1 in categorical columns are passed as-is and treated as their own category, requiring no further processing.

For numeric and binary features, StandardScaler normalization is similarly applied, with the scaler fit on the training set only and applied to validation and test sets to prevent data leakage.

As pytorch-tabular operates on DataFrames, the final inputs are reconstructed by combining scaled numeric and binary features with original categorical columns, aligned using the same train/validation/test indices.

In [28]:
# FT-Transformer needs DataFrames with original unencoded categorical columns
# df still has original values — use split indices to align with the same 60/20/20 split
train_df_ft = df.loc[X_train.index].copy().reset_index(drop=True)
val_df_ft   = df.loc[X_val.index].copy().reset_index(drop=True)
test_df_ft  = df.loc[X_test.index].copy().reset_index(drop=True)

# Apply already-scaled numeric and binary values
for ft_df, split in zip([train_df_ft, val_df_ft, test_df_ft],
                        [X_train, X_val, X_test]):
    ft_df[SCALE_COLS] = split[SCALE_COLS].values

# FT-Transformer expects categoricals as strings
for ft_df in [train_df_ft, val_df_ft, test_df_ft]:
    for col in CAT_COLS:
        ft_df[col] = ft_df[col].astype(str)



**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `attn_heads`: number of attention heads in each transformer block (4 or 8)
- `num_layers`: number of transformer blocks
- `lr`: learning rate

In [29]:
def objective_ft(trial):
    attn_heads = trial.suggest_categorical("attn_heads", [4, 8])
    num_layers  = trial.suggest_int("num_layers", 2, 6)
    lr          = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    data_config = DataConfig(
        target=["target"],
        continuous_cols=NUM_COLS + BIN_COLS,
        categorical_cols=CAT_COLS
    )
    trainer_config = TrainerConfig(
    max_epochs=15,
    early_stopping="valid_loss",
    early_stopping_patience=3,
    checkpoints=None,
    load_best=False,  
    progress_bar="none",
    trainer_kwargs={"enable_model_summary": False}
    )
    optimizer_config = OptimizerConfig()
    model_config = FTTransformerConfig(
        task="classification",
        num_attn_blocks=num_layers,
        num_heads=attn_heads,
        learning_rate=lr
    )
    model_ft = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config
    )
    model_ft.fit(train=train_df_ft, validation=val_df_ft)
    preds = model_ft.predict(val_df_ft)
    prob_cols = [c for c in preds.columns if "probability" in c.lower()]
    if len(prob_cols) == 0:
        raise ValueError(f"No probability column found. Got: {preds.columns.tolist()}")
    preds_proba = preds[prob_cols[-1]].values
    auc = roc_auc_score(val_df_ft["target"].values, preds_proba)
    return auc

study_ft = optuna.create_study(direction="maximize")
study_ft.optimize(objective_ft, n_trials=20)
print("Best AUC:", study_ft.best_value)
print("Best params:", study_ft.best_params)

2026-04-16 19:51:01,151 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-16 19:51:01,186 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-16 19:51:01,340 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-16 19:51:01,788 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-16 19:51:01,888 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-16 19:51:01,908 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-16 20:06:50,133 - {pytorch_tabular.tabular_model:690

Best AUC: 0.6363609101339708
Best params: {'attn_heads': 4, 'num_layers': 4, 'lr': 0.00045385119988985447}


**Final Evaluation**

In [30]:
best_params_ft = study_ft.best_params
seeds = [1, 2, 3]

final_acc_ft, final_auc_ft = [], []

for seed in seeds:
    np.random.seed(seed)
    train_indices = np.random.permutation(len(train_df_ft))
    train_df_shuffled = train_df_ft.iloc[train_indices].reset_index(drop=True)

    data_config = DataConfig(
        target=["target"],
        continuous_cols=NUM_COLS + BIN_COLS,
        categorical_cols=CAT_COLS
    )
    trainer_config = TrainerConfig(
        max_epochs=15,
        early_stopping="valid_loss",
        early_stopping_patience=3,
        checkpoints=None,
        load_best=False,
        progress_bar="none",
        trainer_kwargs={"enable_model_summary": False}
    )
    optimizer_config = OptimizerConfig()
    model_config = FTTransformerConfig(
        task="classification",
        num_attn_blocks=best_params_ft["num_layers"],
        num_heads=best_params_ft["attn_heads"],
        learning_rate=best_params_ft["lr"]
    )
    model_ft = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config
    )

    model_ft.fit(train=train_df_shuffled, validation=val_df_ft)
    preds = model_ft.predict(test_df_ft)

    # safely extract probability column
    prob_cols = [c for c in preds.columns if "probability" in c.lower()]
    if len(prob_cols) == 0:
        raise ValueError(f"No probability column found. Got: {preds.columns.tolist()}")
    preds_proba = preds[prob_cols[-1]].values
    preds_label = (preds_proba >= 0.5).astype(int)

    final_acc_ft.append(accuracy_score(test_df_ft["target"].values, preds_label))
    final_auc_ft.append(roc_auc_score(test_df_ft["target"].values, preds_proba))

    print(f"seed={seed}, acc={final_acc_ft[-1]:.10f}, auc={final_auc_ft[-1]:.10f}")

print(f"FT-Transformer Accuracy: {np.mean(final_acc_ft):.4f} ± {np.std(final_acc_ft):.4f}")
print(f"FT-Transformer AUC-ROC:  {np.mean(final_auc_ft):.4f} ± {np.std(final_auc_ft):.4f}")

2026-04-17 02:48:57,367 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-17 02:48:57,386 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-17 02:48:57,667 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-17 02:48:58,417 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-17 02:48:58,530 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-17 02:48:58,546 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-17 03:02:52,517 - {pytorch_tabular.tabular_model:690

seed=1, acc=0.9635475003, auc=0.6241657544


2026-04-17 03:03:01,255 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-17 03:03:01,274 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-17 03:03:01,537 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-17 03:03:02,273 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-17 03:03:02,359 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-17 03:03:02,371 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-17 03:28:22,408 - {pytorch_tabular.tabular_model:690

seed=2, acc=0.9635475003, auc=0.6214155824


2026-04-17 03:28:31,475 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-17 03:28:32,150 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-17 03:28:32,235 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-17 03:28:32,247 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-17 03:46:07,216 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed


seed=3, acc=0.9635475003, auc=0.6245395917
FT-Transformer Accuracy: 0.9635 ± 0.0000
FT-Transformer AUC-ROC:  0.6234 ± 0.0014


NOTE: 

Since pytorch_tabular internally calls seed_everything(42) regardless of the seed parameter passed to TrainerConfig, we implement an alternative approach: shuffling the training data order with different seeds before training. Even though weight initialization remains locked to seed 42, different batch orderings during training produce results in varying final model weights and predictions.

**Inference Cost**

In [31]:
start = time.time()
_ = model_ft.predict(test_df_ft)
elapsed_ft = time.time() - start

print(f"Inference time per sample: {elapsed_ft/len(test_df_ft)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_ft)) / 1024:.2f} KB")

Inference time per sample: 0.2342 ms
Model size: 57449.72 KB


**Feature Importance**

In [32]:
# manual permutation importance for FT-Transformer
X_test_ft = test_df_ft.drop(columns=["target"]).copy()
y_test_ft  = test_df_ft["target"].values

def get_positive_proba(model, df):
    preds = model.predict(df)
    prob_cols = [c for c in preds.columns if "probability" in c.lower()]
    if len(prob_cols) == 0:
        raise ValueError(f"No probability column found. Got: {preds.columns.tolist()}")
    return preds[prob_cols[-1]].to_numpy().ravel()

# baseline AUC
baseline_proba = get_positive_proba(model_ft, X_test_ft)
baseline_auc   = roc_auc_score(y_test_ft, baseline_proba)

n_repeats = 5
rng = np.random.RandomState(42)

importances_mean = []
importances_all  = {}

for col in X_test_ft.columns:
    drops = []

    for _ in range(n_repeats):
        X_perm = X_test_ft.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)

        perm_proba = get_positive_proba(model_ft, X_perm)
        perm_auc   = roc_auc_score(y_test_ft, perm_proba)

        drops.append(baseline_auc - perm_auc)

    importances_all[col] = drops
    importances_mean.append(np.mean(drops))

ft_importance_series = pd.Series(importances_mean, index=X_test_ft.columns).sort_values(ascending=False)

print("Baseline AUC:", baseline_auc)
print("\nFT-Transformer permutation importances (mean AUC drop):")
print(ft_importance_series)

ft_feature_ranking = ft_importance_series.index.tolist()
print("\nFeature ranking (most to least important):")
print(ft_feature_ranking)

Baseline AUC: 0.6245395917176985

FT-Transformer permutation importances (mean AUC drop):
ps_ind_05_cat     0.008944
ps_ind_15         0.005811
ps_car_15         0.005280
ps_ind_17_bin     0.005273
ps_ind_03         0.004893
ps_reg_03         0.004709
ps_car_13         0.003401
ps_reg_01         0.003358
ps_ind_16_bin     0.003151
ps_car_01_cat     0.002899
ps_reg_02         0.002577
ps_ind_07_bin     0.002459
ps_car_09_cat     0.002330
ps_ind_08_bin     0.001492
ps_car_06_cat     0.001277
ps_ind_06_bin     0.001196
ps_car_11_cat     0.000804
ps_car_12         0.000646
ps_car_05_cat     0.000629
ps_ind_01         0.000588
ps_car_03_cat     0.000342
ps_car_04_cat     0.000311
ps_car_07_cat     0.000246
ps_ind_04_cat     0.000231
ps_calc_04        0.000221
ps_calc_05        0.000219
ps_calc_07        0.000178
ps_calc_09        0.000149
ps_calc_01        0.000146
ps_calc_13        0.000139
ps_calc_18_bin    0.000035
ps_ind_02_cat     0.000015
ps_car_11         0.000014
ps_calc_19_bin    0

**Export Results**

In [33]:
# export obtained results, including feature importance ranking
new_row = {
    "Model": ["FT-Transformer"],
    "Dataset": ["Porto Seguro"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc_ft)],
    "Accuracy Std": [np.std(final_acc_ft)],
    "AUC-ROC Mean": [np.mean(final_auc_ft)],
    "AUC-ROC Std": [np.std(final_auc_ft)],
    "Inference Time (ms/sample)": [elapsed_ft / len(test_df_ft) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model_ft)) / 1024],
    "Feature Ranking (most to least important)": [ft_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_porto_seguro.csv", index=False)

print(results_df)

            Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0          TabNet  Porto Seguro  Classification       0.963548           0.0   
1  FT-Transformer  Porto Seguro  Classification       0.963548           0.0   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.563400     0.010111                    0.013349       427.969727   
1      0.623374     0.001393                    0.234212     57449.715820   

           Feature Ranking (most to least important)  
0  ['ps_ind_06_bin', 'ps_calc_17_bin', 'ps_car_12...  
1  [ps_ind_05_cat, ps_ind_15, ps_car_15, ps_ind_1...  


**XGBoost**
---
For XGBoost, categorical features are one-hot encoded, since `XGBClassifier` expects numeric inputs. Missing values encoded as -1 in categorical columns are treated as their own category and are preserved during one-hot encoding.

XGBoost does not require feature scaling, so numeric and binary features are used in their original form. Missing numerics are retained as -1 and used directly by model.

The same train/validation/test split is used as above, and dummy columns are aligned to the training set to ensure consistent feature representation across splits.

In [34]:
# use original (unscaled) features for XGBoost
train_df_xgb = df.loc[X_train.index].copy().reset_index(drop=True)
val_df_xgb   = df.loc[X_val.index].copy().reset_index(drop=True)
test_df_xgb  = df.loc[X_test.index].copy().reset_index(drop=True)

X_train_xgb_raw = train_df_xgb.drop(columns=["target"])
X_val_xgb_raw   = val_df_xgb.drop(columns=["target"])
X_test_xgb_raw  = test_df_xgb.drop(columns=["target"])

y_train_xgb = train_df_xgb["target"].values
y_val_xgb   = val_df_xgb["target"].values
y_test_xgb  = test_df_xgb["target"].values

# one-hot encode categoricals — -1 becomes its own category column
X_train_xgb = pd.get_dummies(X_train_xgb_raw, columns=CAT_COLS, drop_first=False)
X_val_xgb   = pd.get_dummies(X_val_xgb_raw,   columns=CAT_COLS, drop_first=False)
X_test_xgb  = pd.get_dummies(X_test_xgb_raw,  columns=CAT_COLS, drop_first=False)

# align validation/test columns to training columns
X_val_xgb  = X_val_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)
X_test_xgb = X_test_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:

- n_estimators: number of boosting trees in the ensemble
- max_depth: maximum depth of each decision tree
- learning_rate: step size controlling the contribution of each tree

In [35]:
def objective_xgb(trial):
    params = {
        "n_estimators":  trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth":     trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        "use_label_encoder": False,
        "eval_metric": "auc",
        "random_state": 0,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)
    model.fit(
        X_train_xgb, y_train_xgb,
        eval_set=[(X_val_xgb, y_val_xgb)],
        verbose=False
    )

    preds_proba = model.predict_proba(X_val_xgb)[:, 1]
    auc = roc_auc_score(y_val_xgb, preds_proba)
    return auc

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=20)
print("Best AUC:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

Best AUC: 0.6377030235223691
Best params: {'n_estimators': 400, 'max_depth': 4, 'lr': 0.032540173201372334}


**Final Evaluation**

In [36]:
best_params_xgb = study_xgb.best_params
seeds = [1, 2, 3]

final_acc_xgb, final_auc_xgb = [], []

for seed in seeds:
    model_xgb = XGBClassifier(
        n_estimators=best_params_xgb["n_estimators"],
        max_depth=best_params_xgb["max_depth"],
        learning_rate=best_params_xgb["lr"],
        use_label_encoder=False,
        eval_metric="auc",
        random_state=seed,
        n_jobs=-1,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    model_xgb.fit(X_train_xgb, y_train_xgb, verbose=False)

    preds       = model_xgb.predict(X_test_xgb)
    preds_proba = model_xgb.predict_proba(X_test_xgb)[:, 1]

    final_acc_xgb.append(accuracy_score(y_test_xgb, preds))
    final_auc_xgb.append(roc_auc_score(y_test_xgb, preds_proba))

    print(f"seed={seed}, acc={final_acc_xgb[-1]:.10f}, auc={final_auc_xgb[-1]:.10f}")

print(f"XGBoost Accuracy: {np.mean(final_acc_xgb):.4f} ± {np.std(final_acc_xgb):.4f}")
print(f"XGBoost AUC-ROC:  {np.mean(final_auc_xgb):.4f} ± {np.std(final_auc_xgb):.4f}")

seed=1, acc=0.9635475003, auc=0.6268866113
seed=2, acc=0.9635475003, auc=0.6266351172
seed=3, acc=0.9635475003, auc=0.6250655670
XGBoost Accuracy: 0.9635 ± 0.0000
XGBoost AUC-ROC:  0.6262 ± 0.0008


**Inference Cost**

In [37]:
start = time.time()
_ = model_xgb.predict(X_test_xgb)
elapsed_xgb = time.time() - start

print(f"Inference time per sample: {elapsed_xgb / len(X_test_xgb) * 1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_xgb)) / 1024:.2f} KB")

Inference time per sample: 0.0011 ms
Model size: 648.15 KB


**Feature Importance**

In [38]:
xgb_importance_series = (
    pd.Series(model_xgb.feature_importances_, index=X_train_xgb.columns)
    .sort_values(ascending=False)
)

xgb_feature_ranking = xgb_importance_series.index.tolist()

print(xgb_importance_series.head(20))
print(xgb_feature_ranking)

ps_car_07_cat_-1    0.026866
ps_ind_05_cat_0     0.022822
ps_ind_17_bin       0.014481
ps_ind_07_bin       0.013421
ps_ind_06_bin       0.012928
ps_car_07_cat_1     0.011963
ps_car_03_cat_-1    0.011667
ps_ind_16_bin       0.011594
ps_car_13           0.010964
ps_ind_05_cat_-1    0.010024
ps_car_11_cat_27    0.008331
ps_reg_03           0.007696
ps_reg_02           0.007641
ps_car_01_cat_7     0.007606
ps_car_04_cat_1     0.007078
ps_ind_15           0.006879
ps_ind_05_cat_6     0.006874
ps_car_11_cat_44    0.006871
ps_car_10_cat_0     0.006841
ps_car_08_cat_1     0.006793
dtype: float32
['ps_car_07_cat_-1', 'ps_ind_05_cat_0', 'ps_ind_17_bin', 'ps_ind_07_bin', 'ps_ind_06_bin', 'ps_car_07_cat_1', 'ps_car_03_cat_-1', 'ps_ind_16_bin', 'ps_car_13', 'ps_ind_05_cat_-1', 'ps_car_11_cat_27', 'ps_reg_03', 'ps_reg_02', 'ps_car_01_cat_7', 'ps_car_04_cat_1', 'ps_ind_15', 'ps_ind_05_cat_6', 'ps_car_11_cat_44', 'ps_car_10_cat_0', 'ps_car_08_cat_1', 'ps_ind_03', 'ps_ind_09_bin', 'ps_car_11_cat_93', '

**Export Results**

In [39]:
new_row = {
    "Model": ["XGBoost"],
    "Dataset": ["Porto Seguro"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc_xgb)],
    "Accuracy Std": [np.std(final_acc_xgb)],
    "AUC-ROC Mean": [np.mean(final_auc_xgb)],
    "AUC-ROC Std": [np.std(final_auc_xgb)],
    "Inference Time (ms/sample)": [elapsed_xgb / len(X_test_xgb) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model_xgb)) / 1024],
    "Feature Ranking (most to least important)": [xgb_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_porto_seguro.csv", index=False)

print(results_df)

            Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0          TabNet  Porto Seguro  Classification       0.963548           0.0   
1  FT-Transformer  Porto Seguro  Classification       0.963548           0.0   
2         XGBoost  Porto Seguro  Classification       0.963548           0.0   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.563400     0.010111                    0.013349       427.969727   
1      0.623374     0.001393                    0.234212     57449.715820   
2      0.626196     0.000806                    0.001085       648.150391   

           Feature Ranking (most to least important)  
0  ['ps_ind_06_bin', 'ps_calc_17_bin', 'ps_car_12...  
1  [ps_ind_05_cat, ps_ind_15, ps_car_15, ps_ind_1...  
2  [ps_car_07_cat_-1, ps_ind_05_cat_0, ps_ind_17_...  


**Random Forest**
---
For Random Forest, categorical features are one-hot encoded, since Random Forest expects numeric inputs. Missing values encoded as -1 in categorical columns are treated as their own category and are therefore preserved during one-hot encoding.

Random Forest does not require feature scaling, so numeric and binary features are used in their original unscaled form. Missing numerics are retained as -1 and used directly by model.

The same train/validation/test split is used as above, and dummy columns are aligned to the training set to ensure consistent feature representation across splits.



In [40]:
X_train_rf = X_train_xgb.copy()
X_val_rf   = X_val_xgb.copy()
X_test_rf  = X_test_xgb.copy()
y_train_rf = y_train_xgb.copy()
y_val_rf   = y_val_xgb.copy()
y_test_rf  = y_test_xgb.copy()

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `n_estimators`: number of trees in the forest
- `max_depth`: maximum depth of each decision tree
- `max_features`: number of features considered at each split

In [41]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth":    trial.suggest_int("max_depth", 3, 12),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 0,
        "n_jobs": -1
    }

    model_rf = RandomForestClassifier(**params)
    model_rf.fit(X_train_xgb, y_train_xgb)

    preds_proba = model_rf.predict_proba(X_val_xgb)[:, 1]
    auc = roc_auc_score(y_val_xgb, preds_proba)
    return auc

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=20)
print("Best AUC:", study_rf.best_value)
print("Best params:", study_rf.best_params)

Best AUC: 0.6334910791928812
Best params: {'n_estimators': 300, 'max_depth': 11, 'max_features': 'sqrt'}


**Final Evaluation**

In [42]:
best_params_rf = study_rf.best_params
seeds = [1, 2, 3]

final_acc_rf, final_auc_rf = [], []

for seed in seeds:
    model_rf = RandomForestClassifier(
        **best_params_rf,
        random_state=seed,
        n_jobs=-1
    )

    model_rf.fit(X_train_rf, y_train_rf)

    preds       = model_rf.predict(X_test_rf)
    preds_proba = model_rf.predict_proba(X_test_rf)[:, 1]

    final_acc_rf.append(accuracy_score(y_test_rf, preds))
    final_auc_rf.append(roc_auc_score(y_test_rf, preds_proba))

    print(f"seed={seed}, acc={final_acc_rf[-1]:.10f}, auc={final_auc_rf[-1]:.10f}")

print(f"RF Accuracy: {np.mean(final_acc_rf):.4f} ± {np.std(final_acc_rf):.4f}")
print(f"RF AUC-ROC:  {np.mean(final_auc_rf):.4f} ± {np.std(final_auc_rf):.4f}")

seed=1, acc=0.9635475003, auc=0.6206893823
seed=2, acc=0.9635475003, auc=0.6198878741
seed=3, acc=0.9635475003, auc=0.6171925965
RF Accuracy: 0.9635 ± 0.0000
RF AUC-ROC:  0.6193 ± 0.0015


**Inference Cost**

In [43]:
start = time.time()
_ = model_rf.predict(X_test_rf)
elapsed_rf = time.time() - start

print(f"Inference time per sample: {elapsed_rf / len(X_test_rf) * 1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_rf)) / 1024:.2f} KB")

Inference time per sample: 0.0023 ms
Model size: 23673.76 KB


**Feature Importance**

In [44]:
rf_importance_series = (
    pd.Series(model_rf.feature_importances_, index=X_train_rf.columns)
    .sort_values(ascending=False)
)

rf_feature_ranking = rf_importance_series.index.tolist()

print(rf_importance_series.head(20))
print(rf_feature_ranking)

ps_car_13     0.050299
ps_reg_03     0.043021
ps_reg_02     0.028331
ps_car_14     0.028085
ps_ind_15     0.026706
ps_calc_11    0.026330
ps_calc_14    0.026204
ps_calc_10    0.026123
ps_ind_03     0.024256
ps_calc_13    0.022540
ps_calc_01    0.021135
ps_calc_02    0.021024
ps_calc_03    0.020659
ps_car_12     0.020381
ps_calc_07    0.020325
ps_calc_08    0.020176
ps_calc_09    0.019728
ps_ind_01     0.018921
ps_reg_01     0.018739
ps_calc_06    0.018523
dtype: float64
['ps_car_13', 'ps_reg_03', 'ps_reg_02', 'ps_car_14', 'ps_ind_15', 'ps_calc_11', 'ps_calc_14', 'ps_calc_10', 'ps_ind_03', 'ps_calc_13', 'ps_calc_01', 'ps_calc_02', 'ps_calc_03', 'ps_car_12', 'ps_calc_07', 'ps_calc_08', 'ps_calc_09', 'ps_ind_01', 'ps_reg_01', 'ps_calc_06', 'ps_car_15', 'ps_calc_12', 'ps_calc_05', 'ps_calc_04', 'ps_ind_05_cat_0', 'ps_ind_17_bin', 'ps_ind_07_bin', 'ps_ind_16_bin', 'ps_car_11', 'ps_calc_16_bin', 'ps_calc_17_bin', 'ps_car_07_cat_1', 'ps_ind_06_bin', 'ps_ind_04_cat_1', 'ps_car_01_cat_11', 'ps_

**Export Results**

In [45]:
new_row = {
    "Model": ["Random Forest"],
    "Dataset": ["Porto Seguro"],
    "Task": ["Classification"],
    "Accuracy Mean": [np.mean(final_acc_rf)],
    "Accuracy Std": [np.std(final_acc_rf)],
    "AUC-ROC Mean": [np.mean(final_auc_rf)],
    "AUC-ROC Std": [np.std(final_auc_rf)],
    "Inference Time (ms/sample)": [elapsed_rf / len(X_test_rf) * 1000],
    "Model Size (KB)": [len(pickle.dumps(model_rf)) / 1024],
    "Feature Ranking (most to least important)": [rf_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_porto_seguro.csv", index=False)

print(results_df)

            Model       Dataset            Task  Accuracy Mean  Accuracy Std  \
0          TabNet  Porto Seguro  Classification       0.963548           0.0   
1  FT-Transformer  Porto Seguro  Classification       0.963548           0.0   
2         XGBoost  Porto Seguro  Classification       0.963548           0.0   
3   Random Forest  Porto Seguro  Classification       0.963548           0.0   

   AUC-ROC Mean  AUC-ROC Std  Inference Time (ms/sample)  Model Size (KB)  \
0      0.563400     0.010111                    0.013349       427.969727   
1      0.623374     0.001393                    0.234212     57449.715820   
2      0.626196     0.000806                    0.001085       648.150391   
3      0.619257     0.001496                    0.002348     23673.760742   

           Feature Ranking (most to least important)  
0  ['ps_ind_06_bin', 'ps_calc_17_bin', 'ps_car_12...  
1  [ps_ind_05_cat, ps_ind_15, ps_car_15, ps_ind_1...  
2  [ps_car_07_cat_-1, ps_ind_05_cat_0, ps_ind_17